# Long neuroevolution run on Colab

Runs the headless GA with full-state checkpoints so a run can survive a Colab disconnect. See `colab/README.md` for the local dashboard import steps.

Order: setup cells, run cell, then the resume cell only if the runtime restarts. Do not rerun the first training cell after a disconnect.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Get the code

Use the clone cell if the patched code is pushed to GitHub. Otherwise upload a zip of the project to Drive and run the unzip cell instead. Do not run both.

In [ ]:
!git clone https://github.com/Houssemamor/ARTIFICIAL-LIFE-Simulation-Using-Neuroevolution-and-Genetic-Algorithms.git /content/Deep
%cd /content/Deep
# Alternative: !unzip -q /content/drive/MyDrive/Deep/project.zip -d /content/Deep && cd /content/Deep

In [ ]:
# Colab's torch is already CUDA-enabled: reinstalling the PyPI build would downgrade it to CPU.
!pip install -q pygame-ce==2.5.8 pydantic==2.12.5 pydantic-settings==2.14.2 scipy==1.18.1
import torch
print('torch', torch.__version__, 'cuda', torch.cuda.is_available())

## Config and calibration

`evaluation_steps` and `population.size` dominate runtime. Changing either requires regenerating the fitness reference scales. Keep the seeds fixed so a resume is reproducible.

In [ ]:
import json
cfg = json.load(open('configs/baseline.json'))
cfg['experiment_name'] = 'colab_long'
cfg['mode'] = 'headless'
cfg['evolution']['generations'] = 5000
cfg['evolution']['evaluation_steps'] = 150
cfg['population']['size'] = 120
cfg['experiment']['save_interval'] = 50
cfg['seed'] = {'torch': 11, 'numpy': 12, 'random': 13}
json.dump(cfg, open('colab_config.json', 'w'), indent=2)

from analytics.calibration import run_calibration
run_calibration('colab_config.json', episodes=3,
                steps_per_episode=cfg['evolution']['evaluation_steps'],
                output_path='configs/calibration.json')
print(open('configs/calibration.json').read())

## Start a run

Writes `state.npz`, `history.csv`, `best.npz` and `summary.json` into the Drive run directory.

In [ ]:
import os
RUN_DIR = '/content/drive/MyDrive/Deep/runs/long1'
os.makedirs(RUN_DIR, exist_ok=True)
!python -m experiments.run --config colab_config.json --mode headless --device auto --checkpoint {RUN_DIR}/state.npz --history-csv {RUN_DIR}/history.csv --save-interval 50 --best-genome {RUN_DIR}/best.npz --output {RUN_DIR}/summary.json

## Resume after a disconnect

Re-run the setup cells (mount, code, dependencies, config and calibration), then run this cell. The `--resume` flag restores the population, both RNG streams and the history from `state.npz`.

In [ ]:
!python -m experiments.run --config colab_config.json --mode headless --device auto --checkpoint {RUN_DIR}/state.npz --history-csv {RUN_DIR}/history.csv --save-interval 50 --best-genome {RUN_DIR}/best.npz --output {RUN_DIR}/summary.json --resume

## Export to the local machine

Download the files and place them under `experiments/EXP-HEADLESS-COLAB/` as described in `colab/README.md`, then run the two local commands from that guide to update `dashboard.html`.

In [ ]:
from google.colab import files
for name in ('summary.json', 'best.npz', 'history.csv', 'colab_config.json'):
    files.download(f'{RUN_DIR}/{name}')